Website HTML & Javascript



# Import Modules

In [ ]:
!sudo pip3 install  -U -q geopandas pygeos PyMySQL sqlalchemy sql_magic


In [ ]:
!pip install -U PyMySQL sqlalchemy flask pyngrok geopandas pygeos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import threading

from flask import Flask, render_template, jsonify, request
from pyngrok import ngrok

from sqlalchemy import create_engine
import pandas as pd
import base64
from io import BytesIO
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import requests
import json
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Dataset from NYC Open Data: https://data.cityofnewyork.us/City-Government/Neighborhood-Tabulation-Areas/cpf4-rkhq
!curl 'https://data.cityofnewyork.us/api/geospatial/cpf4-rkhq?method=export&format=GeoJSON' -o nyc-neighborhoods.geojson

# Load the shapefile
df_nyc = gpd.GeoDataFrame.from_file('nyc-neighborhoods.geojson')

In [ ]:
!sudo apt-get install -y -qq  libgeos-dev libproj-dev proj-data proj-bin libgdal-dev libspatialindex-dev
!sudo pip install -U -q shapely rtree pygeos geopandas descartes PyMySQL sqlalchemy sql_magic xlrd

## Setup Flask and ngrok

In [ ]:
os.environ["FLASK_DEBUG"] = "true"
app = Flask(__name__, template_folder = '/content/drive/MyDrive/templates')
port = 5000

# Open a ngrok tunnel to the HTTP server
ngrok_authtoken = os.environ["NGROK_AUTHTOKEN"]
ngrok.set_auth_token(ngrok_authtoken)
public_url = ngrok.connect(port).public_url
print(f" * ngrok tunnel '{public_url}' -> 'http://127.0.0.1:{port}'")

# Update any base URLs to use the public ngrok URL
app.config["BASE_URL"] = public_url

## Setup connection to database


In [ ]:
# Setup a connection to database
conn_string = 'mysql+pymysql://{user}:{password}@{host}/{db}?charset={encoding}'.format(
    host = 'db.ipeirotis.org',
    user = 'student',
    db = 'public',
    password = os.environ["DB_PASSWORD"],
    encoding = 'utf8mb4')
engine = create_engine(conn_string)

# Flask: Create HTML Pages

In [ ]:
# Our website has 2 pages.

# (1) Website Home Page, with search interface
@app.route('/')
def home():
    return render_template('trip_advisor_home_page.html') # The coorsponding file under "Template"

# (2) Shows List of rest, we pass a query with name of the res and do an approximate search
@app.route('/list_res',  methods=['GET'])
def list_res():
  return render_template("list_res_api.html")

# (3) Shows List of Hotels, with query passed to it
@app.route('/hotels',  methods=['GET'])
def list_hotel():
  return render_template("hotels.html")

# (1) Create API Call - Search Query: Search by Name Or Zipcode


### Function `get_res_name`: 如果有Input "search_name" --> Approximate Search
- Function = get_res_name
- Input = search_name
- SQL Approximate = res_name_query
- Response = restaurants


In [ ]:
# Function that connects to database, returns back a list of SQL results.
# It has an optional paramter "search_name"

def get_res_name (search_name=None, start=0, limit=10):
  sql = '''
  SELECT DBA as name, yelp_rating as rating, CUISINE_DESCRIPTION as cuisine, zipcode as zip, STREET as st, latitude as lat, longitude as lon
    FROM public.TEST2_Group_I_restaurant_yelp
'''

  # If we have a search query, we add a condition in SQL
  if search_name:
    sql += " WHERE DBA LIKE %(res_name_query)s OR zipcode LIKE %(res_name_query)s"
    sql += " ORDER BY yelp_rating DESC"

  # This (optional) part limits the number of returned stations
  # The offset part allows us to get "later pages" of results
  # if the limit is not sufficient to get everything back

  sql += f" LIMIT {limit} OFFSET {start}"

  with engine.connect() as con:
    # If there is a search query, we populate the parameter in SQL
    # Since we want to have an approximate query uisng LIKE,
    # we put the wildcard % character before and after the search string
    if search_name:
      params = {"res_name_query": '%'+search_name+'%' }
      restaurants = pd.read_sql(sql, con=con, params=params).dropna()
    else:
      restaurants = pd.read_sql(sql, con=con).dropna()

  return restaurants

### Route (1) **/res_api**

If query --> Call the previous function, return respons

- SQL Approximate = res_name_query


In [ ]:
# API call that returns back a list of res.
  # Pass a query parameter call "name" which we then use to search for res that contain that string in their name

@app.route('/res_api')

def res_api():
  search_query = request.args.get('res_name_query')
  restaurants = get_res_name(search_name = search_query)
    # Call the previous function 'get_res_name'

  # Convert the returned dataframe --> A list of dictionaries
  list_of_res = restaurants.to_dict(orient='records')
  api_results = {"restaurants": list_of_res}

  # We JSON-ify our dictionary and return it as the API response
  return jsonify(api_results)

In [ ]:
# # Function that connects to database, returns back a list of SQL results.
# # It has an optional paramter "search_name"

# @app.route('/search_api')

# def search_query():

#   sql = "SELECT DBA as name, yelp_rating as rating, zipcode as zip, latitude as lat, longitude as lon FROM public.TEST2_Group_I_restaurant_yelp "
#   search_query = request.args.get('name')

#   # If there is a search query, we add a condition in SQL
#   if search_query:
#     sql += " WHERE DBA LIKE %(search_name)s"
#   sql += " LIMIT 500"

#   with engine.connect() as con:
#     # If there is a search query, we populate the parameter in SQL
#     if search_query:
#       params = {"search_name": '%'+search_query+'%' }
#       res_names = pd.read_sql(sql, con=con, params=params).dropna()
#     else:
#       res_names = pd.read_sql(sql, con=con).dropna()

#   list_of_name = res_names.to_dict(orient='records')
#   api_results = {"res_names": list_of_name}

#   return jsonify(api_results)

# (2) Create API Call - Map

## Define Plotting Function `create_res_map_image`

In [ ]:
# This function gets as input a Dataframe with the restaurants and returns back a plot.
  # The plot in encoded in "base64" encoding, so that we can return the image back as part of
  # an API call that returns back JSON data.

def create_res_map_image(restaurants):
  fig, ax = plt.subplots(figsize=(7, 5))

  # Create  map of NYC neighborhoods
  nyc_map = df_nyc.plot(linewidth=0.1, color='White', edgecolor='Gray', ax = ax)

  # Plot the matching restaurants
  ax = restaurants.plot(kind='scatter', x='lon', y='lat', s=3, ax=ax)
  buf = BytesIO()
  fig.savefig(buf, format="png")
  b64encoded = base64.b64encode(buf.getbuffer())

  # Embed the result in the html output.
  image_data = b64encoded.decode("ascii")

  return image_data

## Route (2) `/res_map`

- Call the previous function, Create Resulting Map Plot

In [ ]:
# This API call returns a plot, in JSON format
  # This returns a map of NYC with the restaurants that match the query

@app.route('/res_map',  methods=['GET'])

def res_map():
  search_query = request.args.get('res_name_query')
  restaurants = get_res_name(search_name = search_query)

  # Call the previous function
  image_data = create_res_map_image(restaurants)

  # Create response. Put retrieved data as a list of dictionaries, under key "stations".
  results = {"image_data": "data:image/png;base64," + image_data}

  # JSON-ify our dictionary and return it as the API response
  return jsonify(results)

# Analysis

## (1) Traps - Avoid High Yelp rating BUT High amount of Food Safety Violations

### SQL

In [ ]:
# High Yelp rating BUT High amount of Food Safety Violations
restaurants = '''
  SELECT DBA, YELP_RATING,
      COUNT(doh_restaurants.violations.INSPECTION_ID) as nmbr_of_violations,
      public.TEST2_Group_I_restaurant_yelp.STREET,
      public.TEST2_Group_I_restaurant_yelp.ZIPCODE

  FROM public.TEST2_Group_I_restaurant_yelp
  JOIN doh_restaurants.inspections
      ON public.TEST2_Group_I_restaurant_yelp.CAMIS = doh_restaurants.inspections.CAMIS
  JOIN doh_restaurants.violations
      ON doh_restaurants.inspections.INSPECTION_ID = doh_restaurants.violations.INSPECTION_ID
  JOIN doh_restaurants.violation_codes
      ON doh_restaurants.violations.VIOLATION_CODE = doh_restaurants.violation_codes.VIOLATION_CODE
  WHERE YELP_RATING IS NOT NULL
  AND BORO = 'Manhattan'
  GROUP BY doh_restaurants.inspections.CAMIS
  HAVING nmbr_of_violations > 25
  AND COUNT(doh_restaurants.inspections.INSPECTION_ID) > 10
  ORDER BY YELP_RATING DESC
  LIMIT 8;
'''
results = pd.read_sql(restaurants, con=engine)
results

#(SELECT COUNT(*) FROM doh_restaurants.inspections WHERE ACTION = 'Violations found') / (SELECT COUNT(*) FROM doh_restaurants.inspections) as PERCENT_VIOLATIONS


### Connect to website

In [ ]:
# API call that returns back a list of res.
  # Pass a query parameter call "name" which we then use to search for res that contain that string in their name

@app.route('/traps')

def traps_api():

  sql = '''
    SELECT DBA, YELP_RATING,
        COUNT(doh_restaurants.violations.INSPECTION_ID) as nmbr_of_violations,
        public.TEST2_Group_I_restaurant_yelp.STREET
    FROM public.TEST2_Group_I_restaurant_yelp
    JOIN doh_restaurants.inspections
        ON public.TEST2_Group_I_restaurant_yelp.CAMIS = doh_restaurants.inspections.CAMIS
    JOIN doh_restaurants.violations
        ON doh_restaurants.inspections.INSPECTION_ID = doh_restaurants.violations.INSPECTION_ID
    JOIN doh_restaurants.violation_codes
        ON doh_restaurants.violations.VIOLATION_CODE = doh_restaurants.violation_codes.VIOLATION_CODE
    WHERE YELP_RATING IS NOT NULL
    AND BORO = 'Manhattan'
    GROUP BY doh_restaurants.inspections.CAMIS
    HAVING nmbr_of_violations > 25
    AND COUNT(doh_restaurants.inspections.INSPECTION_ID) > 10
    LIMIT 8;
    '''

  with engine.connect() as con:
      traps = pd.read_sql(sql, con=con).dropna()

  # Convert the returned dataframe --> A list of dictionaries
  list_of_traps = traps.to_dict(orient='records')
  api_results = {"traps": list_of_traps}

  # We JSON-ify our dictionary and return it as the API response
  return jsonify(api_results)

## (2) Cuisine with Highet Yelp Rating

### SQL

In [ ]:
# Average yelp rating across cuisines

avgyelp_cuisine = '''
SELECT DISTINCT CUISINE_DESCRIPTION as Cuisine, AVG(YELP_RATING) as Average_Rating
FROM public.TEST2_Group_I_restaurant_yelp
WHERE YELP_RATING IS NOT NULL
AND BORO = 'Manhattan'
GROUP BY CUISINE_DESCRIPTION
HAVING COUNT(public.TEST2_Group_I_restaurant_yelp.CAMIS) > 25
ORDER BY AVG(YELP_RATING) DESC
LIMIT 8

'''
results = pd.read_sql(avgyelp_cuisine, con=engine)
results


### Connect to website

In [ ]:
@app.route('/cuisine_rank')

def cuisine_rank():

  sql = '''
    SELECT DISTINCT CUISINE_DESCRIPTION as Cuisine, AVG(YELP_RATING) as Average_Rating
    FROM public.TEST2_Group_I_restaurant_yelp
    WHERE YELP_RATING IS NOT NULL
    AND BORO = 'Manhattan'
    GROUP BY CUISINE_DESCRIPTION
    HAVING COUNT(public.TEST2_Group_I_restaurant_yelp.CAMIS) > 25
    ORDER BY AVG(YELP_RATING) DESC
    LIMIT 8
    '''

  with engine.connect() as con:
      cuisine_rank = pd.read_sql(sql, con=con).dropna()

  # Convert the returned dataframe --> A list of dictionaries
  list_of_cuisine_rank = cuisine_rank.to_dict(orient='records')
  api_results = {"cuisine_rank": list_of_cuisine_rank}

  # We JSON-ify our dictionary and return it as the API response
  return jsonify(api_results)

## (3) Watch Out: Cuisine with Highest Food Safety violations

### SQL

In [ ]:
# restaurant violations by cuisine

violations_cuisine = '''
SELECT DISTINCT CUISINE_DESCRIPTION as Cuisine, COUNT(doh_restaurants.violations.INSPECTION_ID) as nmbr_of_violations
FROM public.TEST2_Group_I_restaurant_yelp
JOIN doh_restaurants.inspections ON public.TEST2_Group_I_restaurant_yelp.CAMIS = doh_restaurants.inspections.CAMIS
JOIN doh_restaurants.violations ON doh_restaurants.inspections.INSPECTION_ID = doh_restaurants.violations.INSPECTION_ID
JOIN doh_restaurants.violation_codes ON doh_restaurants.violations.VIOLATION_CODE = doh_restaurants.violation_codes.VIOLATION_CODE
WHERE YELP_RATING IS NOT NULL
AND BORO = 'Manhattan'
GROUP BY CUISINE_DESCRIPTION, doh_restaurants.inspections.CAMIS
ORDER BY nmbr_of_violations DESC
LIMIT 5;
'''
result = pd.read_sql(violations_cuisine, con=engine)
result

### Connect to Website

In [ ]:
@app.route('/violations')

def cuisine_vio_image():

  sql = '''
    SELECT DISTINCT CUISINE_DESCRIPTION as Cuisine, COUNT(doh_restaurants.violations.INSPECTION_ID) as nmbr_of_violations
    FROM public.TEST2_Group_I_restaurant_yelp
    JOIN doh_restaurants.inspections ON public.TEST2_Group_I_restaurant_yelp.CAMIS = doh_restaurants.inspections.CAMIS
    JOIN doh_restaurants.violations ON doh_restaurants.inspections.INSPECTION_ID = doh_restaurants.violations.INSPECTION_ID
    JOIN doh_restaurants.violation_codes ON doh_restaurants.violations.VIOLATION_CODE = doh_restaurants.violation_codes.VIOLATION_CODE
    WHERE YELP_RATING IS NOT NULL
    AND BORO = 'Manhattan'
    GROUP BY CUISINE_DESCRIPTION, doh_restaurants.inspections.CAMIS
    ORDER BY nmbr_of_violations DESC
    LIMIT 5;
    '''

  with engine.connect() as con:
      result = pd.read_sql(sql, con=con).dropna()

  fig, ax = plt.subplots(figsize=(7, 5))
  ax = result.plot(kind = 'barh', x = 'Cuisine')

  buf = BytesIO()
  fig.savefig(buf, format="png")

  # Embed the result in the html output.
  data = base64.b64encode(buf.getbuffer()).decode("ascii")

  # Create response. Put retrieved data as a list of dictionaries, under key "stations".
  results = {"image": data}

  # JSON-ify our dictionary and return it as the API response
  return jsonify(results)


## (4) Recommendations: Highes Rated Restaurants with Lowest Violations

### SQL

In [ ]:
# Recommendations
restaurants = '''
  SELECT DBA, YELP_RATING, zipcode, street, COUNT(doh_restaurants.inspections.INSPECTION_ID) as nmbr_of_violationsfound
  FROM public.TEST2_Group_I_restaurant_yelp
  JOIN doh_restaurants.inspections
      ON public.TEST2_Group_I_restaurant_yelp.CAMIS = doh_restaurants.inspections.CAMIS
  JOIN doh_restaurants.violations
      ON doh_restaurants.inspections.INSPECTION_ID = doh_restaurants.violations.INSPECTION_ID
  JOIN doh_restaurants.violation_codes
      ON doh_restaurants.violations.VIOLATION_CODE = doh_restaurants.violation_codes.VIOLATION_CODE
  WHERE YELP_RATING IS NOT NULL
    AND BORO = 'Manhattan'
    AND YELP_RATING > 4
  GROUP BY doh_restaurants.inspections.CAMIS
  ORDER BY nmbr_of_violationsfound ASC
  LIMIT 10;
'''
results = pd.read_sql(restaurants, con=engine)
results

### Connect to website

In [ ]:
@app.route('/res_rank')

def res_rank():

  sql = '''
    SELECT DBA, YELP_RATING, zipcode, street, COUNT(doh_restaurants.inspections.INSPECTION_ID) as nmbr_of_violationsfound
    FROM public.TEST2_Group_I_restaurant_yelp
    JOIN doh_restaurants.inspections
        ON public.TEST2_Group_I_restaurant_yelp.CAMIS = doh_restaurants.inspections.CAMIS
    JOIN doh_restaurants.violations
        ON doh_restaurants.inspections.INSPECTION_ID = doh_restaurants.violations.INSPECTION_ID
    JOIN doh_restaurants.violation_codes
        ON doh_restaurants.violations.VIOLATION_CODE = doh_restaurants.violation_codes.VIOLATION_CODE
    WHERE YELP_RATING IS NOT NULL
      AND BORO = 'Manhattan'
      AND YELP_RATING > 4
    GROUP BY doh_restaurants.inspections.CAMIS
    ORDER BY nmbr_of_violationsfound ASC
    LIMIT 10
    '''

  with engine.connect() as con:
      res_rank = pd.read_sql(sql, con=con).dropna()

  # Convert the returned dataframe --> A list of dictionaries
  list_of_res_rank = res_rank.to_dict(orient='records')
  api_results = {"res_rank": list_of_res_rank}

  # We JSON-ify our dictionary and return it as the API response
  return jsonify(api_results)

## (5) Crime Map

In [ ]:
# Dataset from NYC Open Data: https://data.cityofnewyork.us/City-Government/Neighborhood-Tabulation-Areas/cpf4-rkhq
!curl 'https://data.cityofnewyork.us/api/geospatial/cpf4-rkhq?method=export&format=GeoJSON' -o nyc-neighborhoods.geojson
    # Load the shapefile
df_nyc = gpd.GeoDataFrame.from_file('nyc-neighborhoods.geojson')
    # Limit the data to only Manhattan neighborhoods
# df_manhattan = df_nyc.query(" boro_name =='Manhattan' ")
    # Create a plot
manhattan_plot = df_nyc.plot(linewidth=0.7, color='White', edgecolor='Black', figsize=(15, 10))


crime_map = '''
SELECT LATITUDE, LONGITUDE, YELP_RATING FROM public.TEST2_Group_I_restaurant_yelp
WHERE YELP_RATING IS NOT NULL
AND Longitude IS NOT NULL
LIMIT 1000;
'''
# AND BORO_NM = 'MANHATTAN'
crimesmap = pd.read_sql(crime_map, con=engine)


crimesmap.plot(kind = 'scatter', y='LATITUDE', x='LONGITUDE', s = 0.5, alpha = 1, figsize = (15,10))
sns.kdeplot(
    data = crimesmap,
    x='LONGITUDE',
    y='LATITUDE',
    shade=True,
    gridsize=100,
    cmap='rainbow',
    alpha=0.5,
    n_levels=50,
    ax = manhattan_plot
)

# Start the webserver


In [ ]:
print(f" * ngrok tunnel '{public_url}' -> 'http://127.0.0.1:{port}'")
app.run(use_reloader=False, port=port)

In [ ]:
# # Plot map API --> Connected to list_res.html

# @app.route('/res_map',  methods=['GET'])
# def res_map():

#     sql = "SELECT DBA as name, yelp_rating as rating, zipcode as zip, latitude as lat, longitude as lon FROM public.TEST2_Group_I_restaurant_yelp LIMIT 500"
#     with engine.connect() as connection:
#         res_loc = pd.read_sql(sql, con=connection)

#     fig, ax = plt.subplots()
#     ax = res_loc.plot(kind='scatter', x='lon', y='lat', figsize=(6,6), ax=ax)

#     buf = BytesIO()
#     fig.savefig(buf, format="png")
#     data = base64.b64encode(buf.getbuffer()).decode("ascii")

#     results = {"image": data}

#     return jsonify(results)